# Independent V4 rerun and audit
New implementation of loading, CNN construction, validation checkpointing, scoring and archive verification. **Seeds 11, 23, 37**, all 12 LOSO readers, real and arbitrary sentence labels: **72 new fits**. Original results are preserved. No model weights or EEG caches are saved.

Same raw 49 × 126 inputs and V4 settings, LR0.0001. Fit10 readers, validate the next reader, test the held-out reader; all110 sentence identities are shared across readers. This is V4's intended familiar-sentence protocol, NOT unseen-sentence generalization. The arbitrary-label control checks whether its score can reflect stimulus-linked shortcuts.

Preflight compares the independent inputs with the old loader and checks identical CNN architecture, initial weights and forward outputs before training. It audits reader separation, labels, duplicates and tested cross-reader-mean signatures. This cannot certify undocumented upstream preprocessing. Expected result footprint roughly1–5MB;72 fits may take several hours depending on GPU. Resume rather than restarting completed fits.

## 1. GPU setup
Mount the account with space and access to the shared Thesis folder. No package installations or new dataset downloads.

In [ ]:
import os, sys, subprocess, json
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
import tensorflow as tf
if not tf.config.list_physical_devices('GPU'):
    print('Section 1a can run on CPU. Sections 2–4 require a GPU runtime.')
from google.colab import drive
drive.mount('/content/drive')
repo = '/content/teco-v4-independent-audit-code'
run_ref = 'v4-independent-audit-v1'
if not os.path.isdir(repo):
    subprocess.run(['git','clone','-q','--branch',run_ref,'--single-branch',
        'https://github.com/parmisbathayan/reproduce-teco-results.git',repo],check=True)
os.chdir(repo)
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip() == subprocess.check_output(['git','rev-parse',run_ref+'^{commit}'],text=True).strip(), 'Wrong code revision.'
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(), 'Use a clean fixed source checkout.'
base = '/content/drive/MyDrive/Thesis/Results/reproduce_yaser_codes/eegonly'
local_dir = '/content/teco-v4-independent-audit-results'
mirror_dir = base + '/v4_independent_audit_11_23_37'
command = [sys.executable,'-u','-B','src/v4_independent_audit.py',
    '--trt-dir','/content/drive/MyDrive/Thesis/Data/TRT_Total',
    '--labels-csv','/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv',
    '--reference-dir',base+'/v4_low_lr_confirmation_45_62',
    '--local-dir',local_dir,'--mirror-dir',mirror_dir]
def stream_run(cmd):
    process = subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end='',flush=True)
        rc = process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
    if rc:
        raise subprocess.CalledProcessError(rc,cmd)
print('Source:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('New result folder:',mirror_dir)

## 1a. Can all-zero inputs help distinguish the classes?
Run before training. Displays shared all-zero sentences and zero counts by reader and sentiment. Also tests a simple rule using ONLY whether the entire input is zero: learn the majority class for zero/nonzero inputs from the same ten training readers as V4; evaluate on the held-out reader. Validation and test labels never fit this rule. Ties predict negative. This is a deterministic diagnostic, not another CNN or a seed search. It runs on CPU, saves nothing, and retains every trial.

A high score would show this flag alone can predict labels; a low score does not rule out more detailed padding/coverage shortcuts or prove what the CNN uses. Missing raw EEG cannot be inferred from zero feature inputs alone. The CNN settings and fixed source revision remain unchanged.

In [ ]:
import sys
sys.path.insert(0, repo + '/src')
import numpy as np
import pandas as pd
from IPython.display import display
from v4_independent_audit import load_independent, score

def zero_input_diagnostic(X, y, readers, ids):
    zero = np.all(X == 0, axis=(2, 3))
    counts = zero.sum(axis=0)
    shared = pd.DataFrame([
        {'sentence_id': int(ids[j]), 'label': 'positive' if y[j] else 'negative',
         'zero_readers': int(counts[j]),
         'readers': ', '.join(readers[i] for i in np.flatnonzero(zero[:, j]))}
        for j in np.flatnonzero(counts >= 2)
    ], columns=['sentence_id', 'label', 'zero_readers', 'readers'])
    by_class = pd.DataFrame([
        {'reader': reader, 'label': label, 'total': int((y == c).sum()),
         'all_zero': int(zero[i, y == c].sum()),
         'zero_percent': float(100 * zero[i, y == c].mean())}
        for i, reader in enumerate(readers)
        for c, label in [(0, 'negative'), (1, 'positive')]])
    rows = []
    for p, reader in enumerate(readers):
        fit = [i for i in range(len(readers)) if i not in (p, (p+1) % len(readers))]
        train_flag = zero[fit].reshape(-1)
        train_y = np.tile(y, len(fit))
        fallback = int(train_y.mean() > .5)
        rule = {}
        for flag in (False, True):
            selected = train_y[train_flag == flag]
            rule[flag] = int(selected.mean() > .5) if len(selected) else fallback
        pred = np.array([rule[bool(flag)] for flag in zero[p]])
        metrics = score(y, np.eye(2)[pred])
        rows.append({'reader': reader, 'macro_f1': metrics['macro_f1'],
                     'accuracy': metrics['accuracy'],
                     'predict_if_zero': rule[True], 'predict_if_nonzero': rule[False]})
    return shared, by_class, pd.DataFrame(rows)

X_check, y_check, readers_check, ids_check, _ = load_independent(
    '/content/drive/MyDrive/Thesis/Data/TRT_Total',
    '/content/drive/MyDrive/Thesis/Data/teco_sentiment_labels_task1.csv')
shared_zero, zero_by_class, zero_rule = zero_input_diagnostic(
    X_check, y_check, readers_check, ids_check)
print('All-zero feature inputs:', int(zero_by_class.all_zero.sum()), 'of', len(readers_check)*len(y_check))
print('Same sentence all zero in two or more readers:')
display(shared_zero)
print('Zero inputs by reader and sentiment:')
display(zero_by_class)
print('Zero/nonzero flag ONLY, same V4 reader splits (0=negative, 1=positive):')
display(zero_rule)
print('Mean across readers:')
display(zero_rule[['macro_f1', 'accuracy']].mean().to_frame('mean'))
print('Balanced constant-class reference: accuracy 0.50; macro-F1 0.3333.')
print('Share this output before starting CNN training if you want us to review it first.')
del X_check


## 2. Independent preflight, no training
Checks the saved V4 data fingerprints/runtime, independently rebuilds labels and inputs, compares input arrays with the old loader, and checks CNN initialization/forward equivalence at all three new seeds. The old loader/model are used only for these checks; new training uses the independent implementation.

Also checks reader/example separation, nonzero duplicates, possible same-sentence mean filling from other readers and missing inputs. All-zero inputs remain in the experiment. If a check fails, share the error; do not bypass it or overwrite old results.

In [ ]:
stream_run(command + ['--prepare-only'])
with open(os.path.join(local_dir,'preflight.json')) as f:
    check = json.load(f)
print('Independent loader matches original:',check['independent_loader_equals_original'])
print('Reader partitions disjoint:',check['reader_example_partitions_disjoint'])
print('All-zero inputs retained:',check['all_zero_inputs_by_reader'])
print('Architecture/initialization checks:',check['architecture_checks'])
print('Limitation:',check['limitation'])

## 3. Run or resume 72 new fits
Real labels first:36 fits. Arbitrary labels next:36 fits, using one balanced sentence-label mapping shared across readers and independent of initialization seeds. Each new model starts from scratch.

The test reader is never passed to fitting or checkpoint selection. The selected epoch uses only validation macro-F1; test predictions are produced afterward. Save all outcomes, including one-class collapse. No automatic rerun or parameter selection based on test scores. Verified jobs resume without training again.

In [ ]:
stream_run(command)

## 4. Verify and compare
Independent confusion counts/macro-F1 are recomputed from every saved prediction. During execution those scores are also cross-checked against sklearn. Histories must agree with the selected validation epoch. Train/validation/test scores help diagnose overfitting or optimization problems.

Compare with original V4 real F1 0.6701 and original V5 control F1 0.6094 descriptively: new initialization seeds are used, and the new control has one independently fixed mapping. This is not a paired rerun of the old random-label mappings and does not provide a formal significance test.

In [ ]:
stream_run(command + ['--verify-only'])
import pandas as pd
from IPython.display import display
print('All new seeds/readers, real and arbitrary labels:')
display(pd.read_csv(os.path.join(local_dir,'summary_overall.csv')))
folds = pd.read_csv(os.path.join(local_dir,'fold_metrics.csv'))
print('Mean train/validation/test macro-F1 by condition:')
display(folds.groupby('condition')[['selected_train_macro_f1','selected_validation_macro_f1','macro_f1']].mean())
print('By initialization seed:')
display(folds.groupby(['condition','seed'])[['macro_f1','accuracy']].mean())
import shutil
backup = shutil.make_archive('/content/v4_independent_audit_results','zip',local_dir)
print('Cloud-runtime backup:',backup,os.path.getsize(backup),'bytes')
print('Drive mirror:',mirror_dir)
print('Confirm these files are visible on the Drive website before ending the runtime. Mounted readback is not server-side persistence proof.')